# Convolutional Neural Network (Konvolüsyonel Sinir Ağı) - Canadian Institute For Advanced Research 10 (CIFAR-10) Dataset

CNN, Convolutional Neural Network (Konvolüsyonel Sinir Ağı) anlamına gelir.
- Özellikle görüntü işleme ve bilgisayarla görme alanlarında yaygın olarak kullanılan derin öğrenme modelleridir.
- Görüntülerdeki özellikleri otomatik olarak öğrenebilir ve bu özellikleri kullanarak sınıflandırma, nesne tanıma ve diğer görevleri gerçekleştirebilirler.

CNN'ler, genellikle birkaç katmandan oluşur:
1. **Convolutional Layer (Konvolüsyon Katmanı)**: Görüntüdeki özellikleri çıkarmak için konvolüsyon işlemi uygular. 
   - Filtreler (kernels) kullanarak görüntüyü tarar ve özellik haritaları oluşturur.
2. **Activation Layer (Aktivasyon Katmanı)**: Genellikle ReLU (Rectified Linear Unit) gibi bir aktivasyon fonksiyonu kullanılır. 
   - Bu katman, konvolüsyon katmanından çıkan değerleri dönüştürerek modelin doğrusal olmayan ilişkileri öğrenmesine yardımcı olur.
3. **Pooling Layer (Havuzlama Katmanı)**: Özellik haritalarını küçültmek ve önemli bilgileri korumak için kullanılır. 
   - En yaygın kullanılan havuzlama yöntemi max pooling'dir, bu yöntem her bölgedeki maksimum değeri alır.
4. **Fully Connected Layer (Tam Bağlantılı Katman)**: Konvolüsyon ve havuzlama katmanlarından çıkan özellikleri kullanarak sınıflandırma yapar. 
   - Tüm nöronlar birbirine bağlıdır.

CNN'ler, genellikle büyük veri setleri ve güçlü hesaplama kaynakları gerektirir, ancak görüntü işleme görevlerinde yüksek doğruluk sağlarlar. Ayrıca, CNN'ler sadece görüntülerle sınırlı değildir; ses tanıma, doğal dil işleme ve diğer alanlarda da kullanılabilirler.

> CNN'ler, derin öğrenme alanında önemli bir rol oynar ve birçok uygulamada başarıyla kullanılmıştır. Örneğin, tıbbi görüntü analizi, otonom araçlar, yüz tanıma sistemleri ve daha fazlasında CNN'ler yaygın olarak kullanılmaktadır.

In [13]:
# 🔹 Core
import torch # Derin öğrenme modelleri oluşturmak ve eğitmek için kullanılan bir kütüphane 
import numpy as np # Bilimsel hesaplamalar için kullanılan bir kütüphane

# 🔹 Neural Network
import torch.nn as nn # Sinir ağı katmanları ve yapıları için kullanılan bir modül
import torch.nn.functional as F # Sinir ağı fonksiyonları için kullanılan bir modül
import torch.optim as optim # Optimizasyon algoritmaları için kullanılan bir modül

# 🔹 Vision
import torchvision # Görüntü işleme ve veri setleri için kullanılan bir kütüphane
from torchvision import transforms # Görüntü dönüşümleri ve veri ön işleme için kullanılan bir modül

# 🔹 Data
from torch.utils.data import DataLoader # Veri setlerini yüklemek ve işlemek için kullanılan bir modül

# 🔹 Visualization
import matplotlib.pyplot as plt # Grafik çizimi için kullanılan bir kütüphane
import seaborn as sns # Veri görselleştirme için kullanılan bir kütüphane

# 🔹 Metrics
from sklearn.metrics import confusion_matrix # Sınıflandırma modellerinin performansını değerlendirmek için kullanılan bir metrik

In [14]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu') # Modelin çalıştırılacağı cihazı belirler. 
# Eğer CUDA destekli bir GPU varsa, model GPU'da çalışır; aksi takdirde CPU'da çalışır.

print(f"Using Device: {device}") # Kullanılan cihazı ekrana yazdırır

Using Device: cpu


CNN'lerin temel bileşenleri:
1. **Evrişim Katmanları**: Girdi görüntüsünden özellik haritalarını çıkarma işlemini uygular.
   - Görüntünün farklı bölgelerindeki özellikleri tanımak için filtreler kullanır.
    - Özellik haritalarını oluşturmak için filtreler veya çekirdekler kullanır, özelliklerin lokaliteye (bölgeye) göre algılanmasına izin verir.
2. **Aktivasyon Fonksiyonları**: Modelin doğrusal olmayan ilişkileri öğrenmesini sağlar ve daha karmaşık özellikleri tanımasına yardımcı olur. (derin öğrenme yeteneğini arttırır) 
    - Genelde 'ReLU' kullanılır.
      - ReLU (Rectified Linear Unit) aktivasyon fonksiyonu, negatif değerleri sıfıra çevirirken pozitif değerleri olduğu gibi bırakır.
3. **Havuzlama Katmanları (Pooling Layers)**: Özellik haritalarını küçültmek ve özellik konumuna daha az duyarlı olmasını sağlar. 
    - Max Pooling veya Average Pooling gibi teknikler kullanılır.
    - En yaygın olarak kullanılan Max Pooling kullanılır. (max pooling = piksellerin ağırlıklarının max olanları alır)
4. **Tam Bağlantılı Katmanları (Fully Connected)**: Özellik haritalarından elde edilen özellikleri sınıflandırmak için kullanılır. 
    - Geleneksel sinir ağlarına (ANN) benzer şekilde çalışır.

> CNN'ler, görüntü sınıflandırma, nesne tanıma, yüz tanıma gibi birçok görsel görevde yaygın olarak kullanılır.
> (Özellik haritalarını otomatik olarak çıkarabilmesi ve özellik konumuna göre genelleme yapabilmesi sebebiyle CNN'ler görsel veri işlemede çok başarılıdır..!)

In [15]:
# Transformasyonlar için CIFAR-10 veri setinin ortalama ve standart sapma değerleri kullanılır. 
# Bu değerler, görüntülerin renk kanallarının (R, G, B) ortalama ve standart sapma değerlerini temsil eder.
cifar10_mean = (0.4914, 0.4822, 0.4465) # CIFAR-10 veri setindeki görüntülerin ortalama renk değerleri (R, G, B) olarak tanımlanır. 
cifar10_std = (0.2470, 0.2435, 0.2616) # CIFAR-10 veri setindeki görüntülerin standart sapma renk değerleri (R, G, B) olarak tanımlanır.

# 🔹 Train transform (augmentation olabilir)
# Veri artırma (augmentation) işlemi yapılmaz, sadece görüntüler tensöre dönüştürülür ve normalleştirilir.
# Bu, modelin eğitim sırasında görüntüleri daha iyi işlemesine ve genelleme yapmasına yardımcı olabilir.
train_transform = transforms.Compose([
    transforms.ToTensor(), # Görüntüleri tensöre dönüştürmek için kullanılan bir dönüşüm. Modelin eğitim sırasında görüntüleri işleyebilmesi için gereklidir.
    # Tensör dönüşümü, görüntüleri PyTorch'un anlayabileceği bir formata dönüştürür.
    transforms.Normalize(
        mean=cifar10_mean, # CIFAR-10 veri setindeki görüntülerin ortalama renk değerleri (R, G, B) olarak tanımlanır.
        std=cifar10_std # CIFAR-10 veri setindeki görüntülerin standart sapma renk değerleri (R, G, B) olarak tanımlanır.
    ) # Normalleştirme işlemi, görüntülerin renk değerlerini belirli bir aralığa getirir.
]) 

# 🔹 Test transform (augmentation olmaz)
# Test verileri için veri artırma (augmentation) işlemi yapılmaz, sadece görüntüler tensöre dönüştürülür ve normalleştirilir.
# Bu, modelin değerlendirme sırasında daha doğru sonuçlar vermesine yardımcı olur.
test_transform = transforms.Compose([
    transforms.ToTensor(), # Görüntüleri tensöre dönüştürmek için kullanılan bir dönüşüm. Modelin değerlendirme sırasında görüntüleri işleyebilmesi için gereklidir.
    transforms.Normalize(
        mean=cifar10_mean, # CIFAR-10 veri setindeki görüntülerin ortalama renk değerleri (R, G, B) olarak tanımlanır.
        std=cifar10_std # CIFAR-10 veri setindeki görüntülerin standart sapma renk değerleri (R, G, B) olarak tanımlanır.
    )
])

CIFAR-10, Canadian Institute For Advanced Research 10 https://www.cs.toronto.edu/~kriz/cifar.html adresinden indirilebilir.

CIFAR-10, 10 sınıfa sahip bir görüntü veri setidir ve genellikle makine öğrenmesi ve derin öğrenme modellerinin performansını değerlendirmek ve farklı algoritmaların karşılaştırılması için bir benchmark olarak hizmet eder. 
- Görüntü sınıflandırma, nesne tanıma ve diğer bilgisayarla görme görevlerinde kullanılır.
- CIFAR-10 veri seti, 10 sınıftan oluşur ve her sınıf 6000 görüntü içerir. Toplamda 60000 görüntü vardır.
- CIFAR-10 veri seti, 60.000 renkli görüntü içerir ve her görüntü 32x32 piksel boyutundadır. 
- CIFAR-10 veri seti, eğitim ve test olmak üzere iki bölüme ayrılmıştır: eğitim seti 50.000 görüntü içerirken, test seti 10.000 görüntü içerir. Her görüntü, aşağıdaki 10 sınıftan birine aittir:
  1. Uçak (airplane)
  2. Araba (automobile)
  3. Kuş (bird)
  4. Kedi (cat)
  5. Geyik (deer)
  6. Köpek (dog)
  7. Kurbağa (frog)
  8. At (horse)
  9. Gemi (ship)
  10. Kamyon (truck)
- CIFAR-10, genellikle eğitim ve test süreçlerinde kullanılan bir veri setidir ve birçok makine öğrenmesi ve derin öğrenme kütüphanesi tarafından desteklenir.
  
> CIFAR-10 veri seti, genellikle derin öğrenme modellerinin performansını değerlendirmek için kullanılır ve özellikle görüntü sınıflandırma görevlerinde yaygın olarak tercih edilir.

In [16]:
data_root = './data' # CIFAR-10 veri setinin indirileceği ve saklanacağı dizin. 
# Bu dizin, veri setinin boyutuna bağlı olarak yeterli depolama alanına sahip olmalıdır.

train_dataset = torchvision.datasets.CIFAR10(
    root=data_root, # CIFAR-10 veri setinin indirileceği ve saklanacağı dizin. 
    train=True, # Eğitim verilerini yüklemek için kullanılan bir parametre. True olarak ayarlandığında, eğitim verileri yüklenir.
    download=True, # Veri seti belirtilen dizinde bulunamazsa, internetten indirir.
    transform=train_transform # Eğitim verileri için uygulanacak dönüşümler.
) # CIFAR-10 veri setinin eğitim verilerini yükler ve belirtilen dönüşümleri uygular. Veri seti belirtilen dizinde bulunmazsa, internetten indirir.

test_dataset = torchvision.datasets.CIFAR10(
    root=data_root, # CIFAR-10 veri setinin indirileceği ve saklanacağı dizin.
    train=False, # Test verilerini yüklemek için kullanılan bir parametre. False olarak ayarlandığında, test verileri yüklenir.
    download=True, # Veri seti belirtilen dizinde bulunamazsa, internetten indirir.
    transform=test_transform # Test verileri için uygulanacak dönüşümler.
) # CIFAR-10 veri setinin test verilerini yükler ve belirtilen dönüşümleri uygular. Veri seti belirtilen dizinde bulunmazsa, internetten indirir.

Files already downloaded and verified
Files already downloaded and verified


In [17]:
batch_size = 128 # Modelin eğitim ve değerlendirme sırasında kullanılacak veri örneklerinin sayısı.
num_workers = 0  # Veri yükleyicisinin (DataLoader) kaç iş parçacığı kullanacağını belirler. 
# 0 olarak ayarlandığında, veri yükleme işlemi ana iş parçacığında gerçekleştirilir. 
# Daha yüksek bir değer, veri yükleme işlemini hızlandırabilir, ancak sistem kaynaklarına bağlı olarak performansı etkileyebilir.

train_loader = DataLoader(
    train_dataset, # Eğitim veri setini yüklemek ve işlemek için kullanılan bir veri yükleyici (DataLoader) oluşturur.
    batch_size=batch_size, # Modelin eğitim sırasında kullanılacak veri örneklerinin sayısı.
    shuffle=True, # Eğitim verilerini karıştırarak modelin daha iyi genelleme yapmasına yardımcı olur.
    num_workers=num_workers, # Veri yükleyicisinin (DataLoader) kaç iş parçacığı kullanacağını belirler.
    pin_memory=torch.cuda.is_available() # CUDA destekli bir GPU varsa, veri yükleyicisinin (DataLoader) bellek pinleme özelliğini etkinleştirir. Bu, veri transferini hızlandırabilir ve modelin eğitim performansını artırabilir.
) # Eğitim veri setini yüklemek ve işlemek için kullanılan bir veri yükleyici (DataLoader) oluşturur. 
# Eğitim verilerini karıştırarak modelin daha iyi genelleme yapmasına yardımcı olur. 

test_loader = DataLoader(
    test_dataset, # Test veri setini yüklemek ve işlemek için kullanılan bir veri yükleyici (DataLoader) oluşturur. 
    batch_size=batch_size, # Modelin değerlendirme sırasında kullanılacak veri örneklerinin sayısı.
    shuffle=False,  # Test verilerini karıştırmamak önemlidir.
    num_workers=num_workers, # Veri yükleyicisinin (DataLoader) kaç iş parçacığı kullanacağını belirler.
    pin_memory=torch.cuda.is_available() # CUDA destekli bir GPU varsa, veri yükleyicisinin (DataLoader) bellek pinleme özelliğini etkinleştirir.
) # Test veri setini yüklemek ve işlemek için kullanılan bir veri yükleyici (DataLoader) oluşturur.

In [18]:
images, labels = next(iter(train_loader)) # Eğitim veri yükleyicisinden (train_loader) bir batch (küme) veri alır.
# next(iter(train_loader)) ifadesi, train_loader veri yükleyicisinden bir batch veri almak için kullanılır.
# Bu, veri yükleyicisinden bir batch veri almak için kullanılan bir yöntemdir.

# images değişkeni, alınan batch içindeki görüntüleri içerir.
# labels değişkeni ise bu görüntülere karşılık gelen etiketleri içerir.
# Bu işlem, modelin eğitim sürecinde kullanılacak veri örneklerini ve etiketleri elde etmek için yapılır.

print(
    f'Image Shape: {images.shape}\n'
    f'Labels Shape: {labels.shape}'
)

# [128, 3, 32, 32] -> [batch_size, channels, height, width]
# Bu çıktı, eğitim veri yükleyicisinden (train_loader) alınan görüntülerin ve etiketlerin şekillerini gösterir.
# Görüntüler 128 örnek içerir (batch_size), her biri 3 renk kanalına (R, G, B) ve 32x32 piksel boyutuna sahiptir. 
# Etiketler de 128 örnek içerir, her biri bir sınıfı temsil eder.
# Bu bilgiler, modelin giriş ve çıkış boyutlarını belirlemek için önemlidir.

# Görüntülerin şekli, modelin ilk katmanında beklenen giriş boyutunu belirler.
# Etiketlerin şekli, modelin son katmanında beklenen çıkış boyutunu belirler.
# Bu bilgiler, modelin mimarisini tasarlarken ve eğitim sürecini optimize ederken kritik öneme sahiptir.

# Görüntülerin ve etiketlerin şekillerini anlamak, modelin doğru şekilde yapılandırılmasını sağlar ve eğitim sürecinin sorunsuz bir şekilde ilerlemesine yardımcı olur.

Image Shape: torch.Size([128, 3, 32, 32])
Labels Shape: torch.Size([128])


In [19]:
# CIFAR-10 veri seti, 10 farklı sınıfa sahip 60.000 renkli görüntü içerir. 
# Bu sınıf, bu veri seti üzerinde çalışmak için uygun bir CNN mimarisi sağlar.

class CIFAR10_CNN(nn.Module): # CIFAR-10 veri seti için bir konvolüsyonel sinir ağı (CNN) sınıfı tanımlar.
    # Sınıf, nn.Module sınıfından türetilir, bu da PyTorch'un sinir ağı yapıları için temel bir sınıftır.
    # Sınıf, CIFAR-10 veri seti için özel olarak tasarlanmış bir konvolüsyonel sinir ağı (CNN) modelini temsil eder.
    # Bu sınıf, modelin katmanlarını tanımlamak ve ileri besleme (forward) işlemini gerçekleştirmek için kullanılır.

    def __init__(self, *args, **kwargs): # Sınıfın yapıcı (constructor) metodunu tanımlar. 
        # Yapıcı metod, sınıfın örneği oluşturulduğunda çağrılır ve modelin katmanlarını tanımlamak için kullanılır.
        # *args ve **kwargs, sınıfın oluşturulurken esnek bir şekilde argüman almasını sağlar. Bu, modelin farklı yapılandırmalarla oluşturulmasına olanak tanır.
        # Yapıcı metod, modelin katmanlarını tanımlamak ve ileri besleme (forward) işlemini gerçekleştirmek için kullanılır.
        super().__init__(*args, **kwargs)

        # 🔹 Block 1
        self.conv1_1 = nn.Conv2d(
            in_channels=3, # Giriş görüntüsünün renk kanallarının sayısı (3) olarak tanımlanır (RGB).
            out_channels=32, # Bu katmandan çıkacak özellik haritalarının sayısı (32) olarak tanımlanır.
            kernel_size=3, # Konvolüsyon işlemi sırasında kullanılacak çekirdek (kernel) boyutunu belirler. 3x3'lük bir çekirdek kullanılır.
            padding=1 # Konvolüsyon işlemi sırasında görüntünün kenarlarında oluşabilecek boyut kaybını önlemek için kullanılan bir parametre. 1 olarak ayarlandığında, görüntünün her kenarına 1 piksel eklenir, böylece konvolüsyon işlemi sırasında boyut korunur.
        )

        self.conv1_2 = nn.Conv2d(
            in_channels=32, # Önceki katmandan gelen özellik haritalarının sayısı (32) olarak tanımlanır.
            out_channels=64, # Bu katmandan çıkacak özellik haritalarının sayısı (64) olarak tanımlanır.
            kernel_size=3, # Konvolüsyon işlemi sırasında kullanılacak çekirdek (kernel) boyutunu belirler. 3x3'lük bir çekirdek kullanılır.
            padding=1 # Konvolüsyon işlemi sırasında görüntünün kenarlarında oluşabilecek boyut kaybını önlemek için kullanılan bir parametre. 1 olarak ayarlandığında, görüntünün her kenarına 1 piksel eklenir, böylece konvolüsyon işlemi sırasında boyut korunur.
        )

        self.pool1 = nn.MaxPool2d(
            kernel_size=2, # Maksimum havuzlama işlemi sırasında kullanılacak çekirdek (kernel) boyutunu belirler. 2x2'lik bir çekirdek kullanılır.
            stride=2 # Maksimum havuzlama işlemi sırasında çekirdeğin kayma adımını belirler. 2 olarak ayarlandığında, çekirdek her seferinde 2 piksel kayar.
        )

        # 🔹 Block 2
        self.conv2_1 = nn.Conv2d(
            in_channels=64, # Önceki katmandan gelen özellik haritalarının sayısı (64) olarak tanımlanır.
            out_channels=128, # Bu katmandan çıkacak özellik haritalarının sayısı (128) olarak tanımlanır.
            kernel_size=3, # Konvolüsyon işlemi sırasında kullanılacak çekirdek (kernel) boyutunu belirler. 3x3'lük bir çekirdek kullanılır.
            padding=1 # Konvolüsyon işlemi sırasında görüntünün kenarlarında oluşabilecek boyut kaybını önlemek için kullanılan bir parametre. 1 olarak ayarlandığında, görüntünün her kenarına 1 piksel eklenir, böylece konvolüsyon işlemi sırasında boyut korunur.
        )

        self.conv2_2 = nn.Conv2d(
            in_channels=128, # Önceki katmandan gelen özellik haritalarının sayısı (128) olarak tanımlanır.
            out_channels=128, # Bu katmandan çıkacak özellik haritalarının sayısı (128) olarak tanımlanır.
            kernel_size=3, # Konvolüsyon işlemi sırasında kullanılacak çekirdek (kernel) boyutunu belirler. 3x3'lük bir çekirdek kullanılır.
            padding=1 # Konvolüsyon işlemi sırasında görüntünün kenarlarında oluşabilecek boyut kaybını önlemek için kullanılan bir parametre. 1 olarak ayarlandığında, görüntünün her kenarına 1 piksel eklenir, böylece konvolüsyon işlemi sırasında boyut korunur.
        )
        
        self.pool2 = nn.MaxPool2d(
            kernel_size=2, # Maksimum havuzlama işlemi sırasında kullanılacak çekirdek (kernel) boyutunu belirler. 2x2'lik bir çekirdek kullanılır.
            stride=2 # Maksimum havuzlama işlemi sırasında çekirdeğin kayma adımını belirler. 2 olarak ayarlandığında, çekirdek her seferinde 2 piksel kayar.
        )

        # 🔹 Fully Connected
        self.fc1 = nn.Linear(
            in_features=128 * 8 * 8, # Konvolüsyon ve havuzlama işlemleri sonucunda elde edilen özellik haritalarının sayısı (128) ile her bir özellik haritasının boyutunun (8x8) çarpımı olarak tanımlanır.
            out_features=256 # Bu katmandan çıkacak özelliklerin sayısı (256) olarak tanımlanır.
        )

        self.fc2 = nn.Linear(
            in_features=256, # Önceki katmandan gelen özelliklerin sayısı (256) olarak tanımlanır.
            out_features=10 # Bu katmandan çıkacak özelliklerin sayısı (10) olarak tanımlanır.
        )

        self.dropout = nn.Dropout(p=0.5) # Dropout katmanı, overfitting'i önlemek için kullanılır. p=0.5, her bir nöronun %50 olasılıkla devre dışı bırakılacağını belirtir.

    def forward(self, x): # Sınıfın ileri besleme (forward) metodunu tanımlar. Bu metod, modelin giriş verilerini alır ve katmanlar aracılığıyla işleyerek çıkış (logits) üretir.
        # İleri besleme (forward) işlemi, modelin katmanları aracılığıyla verilerin nasıl işlendiğini tanımlar.
        # Bu metod, modelin eğitim ve değerlendirme sırasında çağrılır ve modelin tahminler üretmesini sağlar.
        # İleri besleme (forward) işlemi sırasında, veriler konvolüsyon katmanlarından geçirilir, aktivasyon fonksiyonları uygulanır, havuzlama işlemleri gerçekleştirilir ve sonunda tam bağlantılı katmanlardan geçirilerek logits üretilir.
        # Bu metod, modelin mimarisine göre verilerin nasıl işlendiğini tanımlar ve modelin tahminler üretmesini sağlar.
        
        x = F.relu(self.conv1_1(x)) # İlk konvolüsyon katmanından sonra ReLU aktivasyon fonksiyonu uygulanır. Bu, modelin doğrusal olmayan ilişkileri öğrenmesine yardımcı olur.
        x = F.relu(self.conv1_2(x)) # İkinci konvolüsyon katmanından sonra ReLU aktivasyon fonksiyonu uygulanır. Bu, modelin doğrusal olmayan ilişkileri öğrenmesine yardımcı olur.
        x = self.pool1(x) # İlk havuzlama katmanı uygulanır. Bu, özellik haritalarının boyutunu azaltarak modelin daha az parametreye sahip olmasını sağlar ve aynı zamanda önemli özelliklerin korunmasına yardımcı olur.

        x = F.relu(self.conv2_1(x)) # Üçüncü konvolüsyon katmanından sonra ReLU aktivasyon fonksiyonu uygulanır. Bu, modelin doğrusal olmayan ilişkileri öğrenmesine yardımcı olur.
        x = F.relu(self.conv2_2(x)) # Dördüncü konvolüsyon katmanından sonra ReLU aktivasyon fonksiyonu uygulanır. Bu, modelin doğrusal olmayan ilişkileri öğrenmesine yardımcı olur.
        x = self.pool2(x) # İkinci havuzlama katmanı uygulanır. Bu, özellik haritalarının boyutunu azaltarak modelin daha az parametreye sahip olmasını sağlar ve aynı zamanda önemli özelliklerin korunmasına yardımcı olur.

        x = torch.flatten(x, start_dim=1) # Özellik haritalarını tek boyutlu bir vektöre dönüştürür. Bu, tam bağlantılı katmanlara giriş olarak kullanılmak üzere gereklidir.

        x = F.relu(self.fc1(x)) # İlk tam bağlantılı katmandan sonra ReLU aktivasyon fonksiyonu uygulanır. Bu, modelin doğrusal olmayan ilişkileri öğrenmesine yardımcı olur.
        x = self.dropout(x) # Dropout katmanı uygulanır. Bu, overfitting'i önlemeye yardımcı olur.

        logits = self.fc2(x) # İkinci tam bağlantılı katmandan çıkış alınır. Bu, modelin sınıflandırma sonuçlarını üretir.

        return logits

In [20]:
# Modelin oluşturulması ve cihaza taşınması, eğitim sürecinin başlaması için gerekli bir adımdır.
# Modelin oluşturulması ve cihaza taşınması, modelin eğitim sürecinde verimli bir şekilde çalışmasını sağlar.

# Modelin oluşturulması, modelin katmanlarının tanımlanması ve yapılandırılması anlamına gelir.
# Modelin oluşturulması, modelin mimarisini tanımlar ve modelin eğitim sürecinde kullanılacak yapıyı sağlar.

model = CIFAR10_CNN().to(device) # CIFAR10_CNN sınıfından bir model örneği oluşturulur ve belirtilen cihaza (CPU veya GPU) taşınır.
# Bu, modelin eğitim ve değerlendirme sırasında kullanılacak cihazda çalışmasını sağlar.
# CIFAR-10 veri seti için tasarlanmış bu CNN modeli, konvolüsyonel katmanlar, havuzlama katmanları, tam bağlantılı katmanlar ve dropout katmanı içerir.

print(model)

CIFAR10_CNN(
  (conv1_1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv1_2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2_1): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2_2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=8192, out_features=256, bias=True)
  (fc2): Linear(in_features=256, out_features=10, bias=True)
  (dropout): Dropout(p=0.5, inplace=False)
)


In [21]:
criterion = nn.CrossEntropyLoss() # Sınıflandırma problemleri için yaygın olarak kullanılan bir kayıp fonksiyonudur.
# CrossEntropyLoss, modelin tahminleri ile gerçek etiketler arasındaki farkı ölçer ve modelin öğrenme sürecinde bu farkı minimize etmeye çalışır.

optimizer = optim.Adam(
    model.parameters(), # Modelin öğrenme sürecinde güncellenecek parametrelerini belirtir. Bu, modelin ağırlıklarını ve biaslarını içerir.
    lr=1e-3, # Öğrenme oranı (learning rate) olarak 0.001 (1e-3) kullanılır. Bu, modelin ağırlıklarını güncellerken ne kadar büyük adımlar atacağını belirler.
    weight_decay=1e-4 # Ağırlık çürümesi (weight decay) olarak 0.0001 (1e-4) kullanılır. Bu, modelin ağırlıklarının güncellenirken küçük bir ceza uygulanmasını sağlar ve overfitting'i önlemeye yardımcı olabilir.
) # Adam optimizasyon algoritması, modelin parametrelerini güncellemek için kullanılır.
# Adam optimizasyon algoritması, modelin parametrelerini güncellemek için kullanılan bir yöntemdir.
# Adam, öğrenme oranını adaptif olarak ayarlayan bir optimizasyon algoritmasıdır ve genellikle derin öğrenme modellerinde iyi performans gösterir.
# Adam, modelin parametrelerini güncellerken momentum ve RMSProp gibi teknikleri kullanarak daha hızlı ve daha etkili bir şekilde öğrenmeyi sağlar.

In [22]:
def train_model(model, dataloader, criterion, optimizer, device): # Modelin eğitim sürecini gerçekleştiren bir fonksiyon tanımlar. Bu fonksiyon, modelin eğitim moduna geçmesini sağlar, her batch için kayıp değerini hesaplar, gradyanları geri yayar ve modelin parametrelerini günceller.
    # Bu fonksiyon, modelin eğitim sürecini yönetir ve her epoch sonunda ortalama kayıp ve doğruluk değerlerini döndürür.
    # Modelin eğitim sürecinde, her batch için kayıp değeri hesaplanır, gradyanlar geri yayılır ve modelin parametreleri güncellenir.
    
    # Bu fonksiyon, modelin eğitim sürecini yönetmek ve modelin performansını değerlendirmek için kullanılır.
    # model: Eğitilecek konvolüsyonel sinir ağı (CNN) modelini temsil eder.
    # dataloader: Eğitim verilerini yüklemek ve işlemek için kullanılan bir veri yükleyici (DataLoader) nesnesini temsil eder.
    # criterion: Modelin tahminleri ile gerçek etiketler arasındaki kaybı hesaplamak için kullanılan bir kayıp fonksiyonunu temsil eder.    
    # optimizer: Modelin parametrelerini güncellemek için kullanılan bir optimizasyon algoritmasını temsil eder.
    # device: Modelin ve verilerin taşınacağı cihazı (CPU veya GPU) temsil eder.

    model.train() # Modelin eğitim moduna geçmesini sağlar. Bu, dropout ve batch normalization gibi katmanların eğitim sırasında farklı davranmasını sağlar.
    # Modelin eğitim moduna geçmesi, modelin eğitim sırasında doğru şekilde çalışmasını sağlar.

    running_loss = 0.0 # Eğitim sürecinde kaybedilen toplam kayıp değerini tutar. Her batch için hesaplanan kayıp değerleri bu değişkende toplanır.
    correct = 0 # Eğitim sürecinde doğru sınıflandırılan örneklerin sayısını tutar. Her batch için doğru sınıflandırılan örnekler bu değişkende toplanır.
    total = 0 # Eğitim sürecinde işlenen toplam örnek sayısını tutar. Her batch için işlenen örnekler bu değişkende toplanır.

    for batch_idx, (inputs, targets) in enumerate(dataloader): # Eğitim veri yükleyicisinden (dataloader) her batch için giriş verilerini (inputs) ve hedef etiketleri (targets) alır.
        inputs = inputs.to(device) # Giriş verilerini belirtilen cihaza (CPU veya GPU) taşır. Bu, modelin eğitim sürecinde verileri işleyebilmesi için gereklidir.
        targets = targets.to(device) # Hedef etiketleri belirtilen cihaza (CPU veya GPU) taşır. Bu, modelin eğitim sürecinde hedef etiketleri işleyebilmesi için gereklidir.

        optimizer.zero_grad() # Optimizatörün (optimizer) gradyanlarını sıfırlar. Bu, her batch için gradyanların birikmesini önler ve modelin doğru şekilde güncellenmesini sağlar.

        outputs = model(inputs) # Modelin ileri besleme (forward) metodunu çağırarak giriş verilerini işler ve tahminler (logits) üretir.
        loss = criterion(outputs, targets) # Modelin tahminleri (outputs) ile gerçek etiketler (targets) arasındaki kaybı hesaplar. Bu, modelin öğrenme sürecinde minimize etmeye çalıştığı bir değerdir.

        loss.backward() # Geri yayılım (backward) işlemini gerçekleştirir. Bu, kaybın modelin parametrelerine göre türevini hesaplar.
        optimizer.step() # Optimizatörün (optimizer) adımını atar. Bu, modelin parametrelerini günceller ve öğrenme sürecini ilerletir.

        running_loss += loss.item() * inputs.size(0) # Her batch için kayıp değerini running_loss değişkenine ekler. loss.item() ifadesi, kayıp değerini bir Python sayısına dönüştürür ve inputs.size(0) ifadesi, batch içindeki örnek sayısını verir. Bu, toplam kayıp değerini hesaplamak için kullanılır.

        _, predicted = outputs.max(1) # Modelin tahminlerini (outputs) alır ve her örnek için en yüksek olasılığa sahip sınıfı belirler. predicted değişkeni, modelin tahmin ettiği sınıf indekslerini içerir.

        total += targets.size(0) # Her batch için işlenen örnek sayısını total değişkenine ekler. targets.size(0) ifadesi, batch içindeki örnek sayısını verir. Bu, toplam işlenen örnek sayısını hesaplamak için kullanılır.
        correct += (predicted == targets).sum().item() # Modelin tahmin ettiği sınıfların (predicted) gerçek etiketlerle (targets) eşleşip eşleşmediğini kontrol eder ve doğru sınıflandırılan örneklerin sayısını correct değişkenine ekler. (predicted == targets) ifadesi, her örnek için tahmin edilen sınıfın gerçek sınıf ile eşleşip eşleşmediğini kontrol eder ve sum() ifadesi, doğru sınıflandırılan örneklerin toplam sayısını verir.

        """ print(
            f'Batch: {batch_idx}\n'
            f'Loss: {loss.item():.4f}\n'
            f'Accuracy: {(correct/total):.4f}\n'
            f'--------------------------'
        )  """
        

    epoch_loss = running_loss / total # Eğitim sürecinde kaybedilen toplam kayıp değerini, işlenen toplam örnek sayısına bölerek ortalama kayıp değerini hesaplar. Bu, modelin eğitim sürecindeki performansını değerlendirmek için kullanılır.
    epoch_acc = correct / total # Eğitim sürecinde doğru sınıflandırılan örneklerin oranını hesaplar. Bu, modelin eğitim sürecindeki doğruluk performansını değerlendirmek için kullanılır.

    return epoch_loss, epoch_acc

In [23]:
def evaluate(model, dataloader, criterion, device): # Modelin değerlendirme sürecini gerçekleştiren bir fonksiyon tanımlar. Bu fonksiyon, modelin değerlendirme moduna geçmesini sağlar, her batch için kayıp değerini hesaplar ve modelin doğruluk performansını ölçer.
    # Bu fonksiyon, modelin değerlendirme sürecini yönetir ve ortalama kayıp ve doğruluk değerlerini döndürür.
    # Modelin değerlendirme sürecinde, her batch için kayıp değeri hesaplanır ve modelin doğruluk performansı ölçülür.
    # model: Değerlendirilecek konvolüsyonel sinir ağı (CNN) modelini temsil eder.
    # dataloader: Değerlendirme verilerini yüklemek ve işlemek için kullanılan bir veri yükleyici (DataLoader) nesnesini temsil eder.
    # criterion: Modelin tahminleri ile gerçek etiketler arasındaki kaybı hesaplamak için kullanılan bir kayıp fonksiyonunu temsil eder.
    # device: Modelin ve verilerin taşınacağı cihazı (CPU veya GPU) temsil eder.
    
    model.eval() # Modelin değerlendirme moduna geçmesini sağlar. Bu, dropout ve batch normalization gibi katmanların değerlendirme sırasında farklı davranmasını sağlar.

    running_loss = 0.0 # Değerlendirme sürecinde kaybedilen toplam kayıp değerini tutar. Her batch için hesaplanan kayıp değerleri bu değişkende toplanır.
    correct = 0 # Değerlendirme sürecinde doğru sınıflandırılan örneklerin sayısını tutar. Her batch için doğru sınıflandırılan örnekler bu değişkende toplanır.
    total = 0 # Değerlendirme sürecinde işlenen toplam örnek sayısını tutar. Her batch için işlenen örnekler bu değişkende toplanır.

    with torch.no_grad(): # Geri yayılımın (backward) hesaplanmasını devre dışı bırakır. Bu, değerlendirme sürecinde gereksiz hesaplamaları önler ve bellek kullanımını azaltır.
        for inputs, targets in dataloader: # Değerlendirme sürecinde her batch için giriş verilerini (inputs) ve hedef etiketleri (targets) alır.
            inputs = inputs.to(device) # Giriş verilerini belirtilen cihaza (CPU veya GPU) taşır. Bu, modelin değerlendirme sürecinde verileri işleyebilmesi için gereklidir.
            targets = targets.to(device) # Hedef etiketleri belirtilen cihaza (CPU veya GPU) taşır. Bu, modelin değerlendirme sürecinde hedef etiketleri işleyebilmesi için gereklidir.

            outputs = model(inputs) # Modelin ileri besleme (forward) metodunu çağırarak giriş verilerini işler ve tahminler (logits) üretir.
            loss = criterion(outputs, targets) # Modelin tahminleri (outputs) ile gerçek etiketler (targets) arasındaki kaybı hesaplar. Bu, modelin değerlendirme sürecinde performansını ölçmek için kullanılır.

            running_loss += loss.item() * inputs.size(0) # Her batch için kayıp değerini running_loss değişkenine ekler. loss.item() ifadesi, kayıp değerini bir Python sayısına dönüştürür ve inputs.size(0) ifadesi, batch içindeki örnek sayısını verir. Bu, toplam kayıp değerini hesaplamak için kullanılır.

            _, predicted = outputs.max(1) # Modelin tahminlerini (outputs) alır ve her örnek için en yüksek olasılığa sahip sınıfı belirler. predicted değişkeni, modelin tahmin ettiği sınıf indekslerini içerir.

            total += targets.size(0) # Her batch için işlenen örnek sayısını total değişkenine ekler. targets.size(0) ifadesi, batch içindeki örnek sayısını verir. Bu, toplam işlenen örnek sayısını hesaplamak için kullanılır.
            correct += (predicted == targets).sum().item() # Modelin tahmin ettiği sınıfların (predicted) gerçek etiketlerle (targets) eşleşip eşleşmediğini kontrol eder ve doğru sınıflandırılan örneklerin sayısını correct değişkenine ekler. (predicted == targets) ifadesi, her örnek için tahmin edilen sınıfın gerçek sınıf ile eşleşip eşleşmediğini kontrol eder ve sum() ifadesi, doğru sınıflandırılan örneklerin toplam sayısını verir.

    epoch_loss = running_loss / total # Değerlendirme sürecinde kaybedilen toplam kayıp değerini, işlenen toplam örnek sayısına bölerek ortalama kayıp değerini hesaplar. Bu, modelin değerlendirme sürecindeki performansını ölçmek için kullanılır.
    epoch_acc = correct / total # Değerlendirme sürecinde doğru sınıflandırılan örneklerin oranını hesaplar. Bu, modelin değerlendirme sürecindeki doğruluk performansını ölçmek için kullanılır.

    return epoch_loss, epoch_acc

In [24]:
NUM_EPOCHS = 10 # Modelin eğitim sürecinde kaç epoch (tam veri seti üzerinden geçiş) gerçekleştirileceğini belirler.

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_acc = train_model(
        model=model, # Eğitilecek konvolüsyonel sinir ağı (CNN) modelini temsil eder.
        dataloader=train_loader, # Eğitim verilerini yüklemek ve işlemek için kullanılan bir veri yükleyici (DataLoader) nesnesini temsil eder.
        criterion=criterion, # Modelin tahminleri ile gerçek etiketler arasındaki kaybı hesaplamak için kullanılan bir kayıp fonksiyonunu temsil eder.
        optimizer=optimizer, # Modelin parametrelerini güncellemek için kullanılan bir optimizasyon algoritmasını temsil eder.
        device=device # Modelin ve verilerin taşınacağı cihazı (CPU veya GPU) temsil eder.
    ) # Modelin eğitim sürecini gerçekleştirir ve her epoch sonunda ortalama kayıp ve doğruluk değerlerini döndürür.

    val_loss, val_acc = evaluate(
        model=model, # Değerlendirilecek konvolüsyonel sinir ağı (CNN) modelini temsil eder.
        dataloader=test_loader, # Değerlendirme verilerini yüklemek ve işlemek için kullanılan bir veri yükleyici (DataLoader) nesnesini temsil eder.
        criterion=criterion, # Modelin tahminleri ile gerçek etiketler arasındaki kaybı hesaplamak için kullanılan bir kayıp fonksiyonunu temsil eder.
        device=device # Modelin ve verilerin taşınacağı cihazı (CPU veya GPU) temsil eder.
    ) # Modelin değerlendirme sürecini gerçekleştirir ve ortalama kayıp ve doğruluk değerlerini döndürür.

    print(
        f'Epoch [{epoch:02}/{NUM_EPOCHS}] | '
        f'Training Loss: {train_loss:.4f} | '
        f'Training Accuracy: {train_acc*100:.2f}% | '
        f'Validation Loss: {val_loss:.4f} | '
        f'Validation Accuracy: {val_acc*100:.2f}%'
    ) # Her epoch sonunda eğitim ve doğrulama kayıp ve doğruluk değerlerini ekrana yazdırır.

Epoch [01/10] | Training Loss: 1.5078 | Training Accuracy: 44.87% | Validation Loss: 1.1641 | Validation Accuracy: 58.84%
Epoch [02/10] | Training Loss: 1.0742 | Training Accuracy: 61.97% | Validation Loss: 0.9317 | Validation Accuracy: 67.13%
Epoch [03/10] | Training Loss: 0.8932 | Training Accuracy: 68.36% | Validation Loss: 0.8156 | Validation Accuracy: 71.86%
Epoch [04/10] | Training Loss: 0.7708 | Training Accuracy: 73.03% | Validation Loss: 0.7419 | Validation Accuracy: 74.84%
Epoch [05/10] | Training Loss: 0.6767 | Training Accuracy: 76.34% | Validation Loss: 0.6796 | Validation Accuracy: 76.39%
Epoch [06/10] | Training Loss: 0.5997 | Training Accuracy: 79.16% | Validation Loss: 0.6780 | Validation Accuracy: 77.28%
Epoch [07/10] | Training Loss: 0.5359 | Training Accuracy: 81.12% | Validation Loss: 0.6512 | Validation Accuracy: 77.69%
Epoch [08/10] | Training Loss: 0.4854 | Training Accuracy: 82.99% | Validation Loss: 0.6538 | Validation Accuracy: 78.33%
Epoch [09/10] | Training

In [25]:
def get_test_predictions(model, dataloader, device): # Modelin test verileri üzerindeki tahminlerini (logits) ve gerçek etiketleri döndüren bir fonksiyon tanımlar. Bu fonksiyon, modelin değerlendirme moduna geçmesini sağlar, her batch için tahminler üretir ve bu tahminleri ve gerçek etiketleri birleştirerek döndürür.
    # Bu fonksiyon, modelin test verileri üzerindeki tahminlerini (logits) ve gerçek etiketleri döndürür. Modelin değerlendirme moduna geçmesini sağlar, her batch için tahminler üretir ve bu tahminleri ve gerçek etiketleri birleştirerek döndürür.
    # model: Test verileri üzerinde tahminler üretmek için kullanılacak konvolüsyonel sinir ağı (CNN) modelini temsil eder.
    # dataloader: Test verilerini yüklemek ve işlemek için kullanılan bir veri yükleyici (DataLoader) nesnesini temsil eder.
    # device: Modelin ve verilerin taşınacağı cihazı (CPU veya GPU) temsil eder.
    
    model.eval() # Modeli değerlendirme moduna geçirir. Bu, dropout ve batch normalization gibi katmanların davranışını değiştirir.

    all_logits = [] # Modelin tahminlerini (logits) depolamak için kullanılan bir listeyi temsil eder.
    all_labels = [] # Gerçek etiketleri depolamak için kullanılan bir listeyi temsil eder.

    with torch.no_grad(): # Değerlendirme sırasında gradyan hesaplamalarını devre dışı bırakır. Bu, bellek kullanımını azaltır ve hesaplamayı hızlandırır.
        for inputs, targets in dataloader: # Değerlendirme sürecinde her batch için giriş verilerini (inputs) ve hedef etiketleri (targets) alır.
            inputs = inputs.to(device) # Giriş verilerini belirtilen cihaza (CPU veya GPU) taşır. Bu, modelin değerlendirme sürecinde verileri işleyebilmesi için gereklidir.
            targets = targets.to(device) # Hedef etiketleri belirtilen cihaza (CPU veya GPU) taşır. Bu, modelin değerlendirme sürecinde hedef etiketleri işleyebilmesi için gereklidir.

            outputs = model(inputs) # Modelin ileri besleme (forward) metodunu çağırarak giriş verilerini işler ve tahminler (logits) üretir.

            all_logits.append(outputs) # Her batch için üretilen tahminleri (logits) all_logits listesine ekler. Bu, tüm test verileri için modelin tahminlerini depolamak için kullanılır.
            all_labels.append(targets) # Her batch için gerçek etiketleri all_labels listesine ekler. Bu, tüm test verileri için gerçek etiketleri depolamak için kullanılır.

    all_logits = torch.cat(all_logits, dim=0) # all_logits listesindeki tüm tahminleri tek bir tensöre birleştirir. dim=0, birleştirme işleminin satır bazında yapılacağını belirtir. Bu, tüm test verileri için modelin tahminlerini tek bir tensör olarak elde etmek için kullanılır.
    all_labels = torch.cat(all_labels, dim=0) # all_labels listesindeki tüm gerçek etiketleri tek bir tensöre birleştirir. dim=0, birleştirme işleminin satır bazında yapılacağını belirtir. Bu, tüm test verileri için gerçek etiketleri tek bir tensör olarak elde etmek için kullanılır.

    return all_logits, all_labels

In [26]:
logits, labels = get_test_predictions(model, test_loader, device) # Modelin test verileri üzerindeki tahminlerini (logits) ve gerçek etiketleri döndürür. Modelin değerlendirme moduna geçmesini sağlar, her batch için tahminler üretir ve bu tahminleri ve gerçek etiketleri birleştirerek döndürür.
# logits değişkeni, modelin test verileri üzerindeki tahminlerini (logits) içerir. labels değişkeni ise modelin test verileri üzerindeki gerçek etiketleri içerir.
# Bu tahminler ve gerçek etiketler, modelin test verileri üzerindeki performansını değerlendirmek ve analiz etmek için kullanılabilir.

print(f'Logits Shape: {logits.shape}') # Modelin test verileri üzerindeki tahminlerinin (logits) şekli ekrana yazdırılır.
print(f'Labels Shape: {labels.shape}') # Modelin test verileri üzerindeki gerçek etiketlerinin şekli ekrana yazdırılır.

Logits Shape: torch.Size([10000, 10])
Labels Shape: torch.Size([10000])


In [27]:
probs = F.softmax(logits, dim=1) # Modelin test verileri üzerindeki tahminlerini (logits) olasılıklara dönüştürür. dim=1, softmax işleminin sınıf boyutu boyunca uygulanacağını belirtir. Bu, modelin her sınıf için tahmin ettiği olasılıkları elde etmek için kullanılır.
# logits, modelin test verileri üzerindeki tahminlerini içerir ve bu tahminler genellikle sınıf skorları veya logit değerleri olarak adlandırılır. F.softmax fonksiyonu, bu logit değerlerini olasılıklara dönüştürür, böylece her sınıf için tahmin edilen olasılıkları elde ederiz. Bu olasılıklar, modelin her sınıf için ne kadar güvenli olduğunu gösterir ve genellikle sınıflandırma problemlerinde kullanılır.
# Bu olasılıklar, modelin test verileri üzerindeki performansını değerlendirmek ve analiz etmek için kullanılabilir. Örneğin, en yüksek olasılığa sahip sınıfı belirlemek veya ROC eğrisi gibi metrikler hesaplamak için bu olasılıkları kullanabiliriz.
# Modelin test verileri üzerindeki tahminlerini (logits) olasılıklara dönüştürmek, modelin performansını daha iyi anlamak ve değerlendirmek için önemli bir adımdır.

y_true = labels.cpu().numpy() # Gerçek etiketleri CPU'ya taşıyarak NumPy dizisine dönüştürür. Bu, scikit-learn gibi kütüphanelerle uyumlu hale getirmek için yapılır.
y_pred = probs.argmax(dim=1).cpu().numpy() # Tahmin edilen sınıf indekslerini belirler, CPU'ya taşıyarak NumPy dizisine dönüştürür. Bu, modelin test verileri üzerindeki tahminlerini sınıf indekslerine dönüştürmek ve scikit-learn gibi kütüphanelerle uyumlu hale getirmek için yapılır.